# GPT 멀티 펑션콜링 (Responses API, Parallel Function Calling)

OpenAI **Responses API**에서는 모델이 한 번의 응답에 **여러 개의 `function_call` 항목**을 반환할 수 있습니다 (`parallel_tool_calls=True`, 기본 활성화).

핵심 규칙:
- `response.output`에서 `type == "function_call"` 항목들을 골라 실행
- `response.output` 전체를 다음 요청의 `input`에 그대로 이어붙인 뒤, 각 호출에 대해 `function_call_output`(같은 `call_id`)을 추가
- `function_call`이 더 이상 없으면 `response.output_text`가 최종 답변

In [1]:
import json

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # .env 의 OPENAI_API_KEY 로드

client = OpenAI()
MODEL = "gpt-5-nano"

## 1. 도구 정의

Responses API의 함수 도구는 `type: "function"`에 `name`/`parameters`가 **평평하게(flat)** 들어갑니다 (Chat Completions의 `function: {...}` 중첩 구조와 다름).
`strict: True` + `additionalProperties: False`를 쓰면 인자가 스키마를 정확히 따르는 것이 보장됩니다.

In [2]:
TOOLS = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "지정한 도시의 현재 날씨와 기온을 조회한다. 사용자가 날씨나 기온을 물으면 이 도구를 호출한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "도시 이름 (영문), 예: Seoul, Tokyo, Paris"}
            },
            "required": ["city"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_exchange_rate",
        "description": "USD 기준 환율을 조회한다. 사용자가 환율이나 통화 변환을 물으면 이 도구를 호출한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "currency": {"type": "string", "description": "통화 코드, 예: KRW, JPY, EUR"}
            },
            "required": ["currency"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

## 2. 도구 구현 (모의 데이터)

Claude 노트북(`notebooks/claude/`)과 동일한 시나리오로, 두 API의 패턴을 비교하기 좋게 맞췄습니다.

In [3]:
MOCK_WEATHER = {
    "Seoul": {"condition": "맑음", "temp_c": 31},
    "Tokyo": {"condition": "흐림", "temp_c": 29},
    "Paris": {"condition": "비", "temp_c": 22},
}
MOCK_RATES = {"KRW": 1385.2, "JPY": 157.8, "EUR": 0.92}


def get_weather(city: str) -> dict:
    if city not in MOCK_WEATHER:
        raise ValueError(f"'{city}' 날씨 정보 없음. 가능한 도시: {list(MOCK_WEATHER)}")
    return {"city": city, **MOCK_WEATHER[city]}


def get_exchange_rate(currency: str) -> dict:
    if currency not in MOCK_RATES:
        raise ValueError(f"'{currency}' 환율 정보 없음. 가능한 통화: {list(MOCK_RATES)}")
    return {"base": "USD", "currency": currency, "rate": MOCK_RATES[currency]}


TOOL_FUNCTIONS = {"get_weather": get_weather, "get_exchange_rate": get_exchange_rate}

## 3. 에이전트 루프

`response.output` 전체를 `input`에 이어붙여 히스토리를 유지하고, 각 `function_call`마다 `function_call_output`을 추가합니다.
실행이 실패해도 결과를 빼먹지 말고 에러 문자열을 `output`으로 돌려줘야 모델이 스스로 복구할 수 있습니다.

In [4]:
def run_agent(user_message: str) -> str:
    input_list = [{"role": "user", "content": user_message}]

    while True:
        response = client.responses.create(
            model=MODEL,
            input=input_list,
            tools=TOOLS,
            parallel_tool_calls=True,  # 기본값이지만 명시
        )

        # function_call 항목을 포함한 출력 전체를 히스토리에 보존
        input_list += response.output

        function_calls = [item for item in response.output if item.type == "function_call"]
        if not function_calls:
            return response.output_text

        print(f"[function_call {len(function_calls)}건]")
        for call in function_calls:
            print(f"  → {call.name}({call.arguments})")
            try:
                args = json.loads(call.arguments)
                result = json.dumps(TOOL_FUNCTIONS[call.name](**args), ensure_ascii=False)
            except Exception as e:
                result = f"Error: {e}"
            input_list.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": result,
            })

## 4. 실행

서로 독립적인 조회 6건(날씨 3 + 환율 3)이므로, 한 턴에 여러 `function_call`이 병렬로 나오는 것을 볼 수 있습니다.

In [5]:
answer = run_agent(
    "서울, 도쿄, 파리의 현재 날씨를 알려주고, 1 USD가 원화·엔화·유로로 각각 얼마인지도 알려줘."
)
print("\n=== 최종 답변 ===")
print(answer)

[function_call 6건]
  → get_weather({"city":"Seoul"})
  → get_weather({"city":"Tokyo"})
  → get_weather({"city":"Paris"})
  → get_exchange_rate({"currency":"KRW"})
  → get_exchange_rate({"currency":"JPY"})
  → get_exchange_rate({"currency":"EUR"})

=== 최종 답변 ===
다음은 요청하신 내용입니다.

- 서울: 맑음, 기온 31°C
- 도쿄: 흐림, 기온 29°C
- 파리: 비, 기온 22°C

환율 (1 USD 기준):
- 1 USD = 1,385.2 KRW
- 1 USD = 157.8 JPY
- 1 USD = 0.92 EUR

추가로 원하시면 특정 도시의 날씨를 더 자세히 보거나, 다른 통화로 환산해 드릴게요.


## 참고

- **Chat Completions와의 차이**: 구 API에서는 `tool_calls` 배열이 assistant 메시지 안에 중첩되고, 결과를 `role: "tool"` 메시지로 반환합니다. Responses API는 `function_call` / `function_call_output`이 최상위 항목으로 평평하게 들어가는 구조입니다.
- **`tool_choice`**: `"auto"`(기본) 외에 `"required"`(반드시 호출), `{"type": "function", "name": "get_weather"}`(특정 도구 강제)를 쓸 수 있습니다.
- **`parallel_tool_calls=False`**: 한 턴에 도구를 하나씩만 호출하게 제한할 수 있습니다.